In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
print("Libraries loaded ✓")

Matplotlib is building the font cache; this may take a moment.


Libraries loaded ✓


In [2]:
df = pd.read_csv('../data/data.csv', encoding='latin-1')
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()



Dataset shape: (541909, 8)
Rows: 541,909
Columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [3]:
print("=== COLUMN DATA TYPES ===")
print(df.dtypes)
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== BASIC STATS ===")
df.describe()

=== COLUMN DATA TYPES ===
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

=== MISSING VALUES ===
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

=== BASIC STATS ===


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [4]:
# Convert date to proper datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Extract useful time columns
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['MonthName'] = df['InvoiceDate'].dt.strftime('%B')
df['DayOfWeek'] = df['InvoiceDate'].dt.strftime('%A')
df['Hour'] = df['InvoiceDate'].dt.hour

print("Date columns created ✓")
df[['InvoiceDate','Year','Month','MonthName','DayOfWeek','Hour']].head()


Date columns created ✓


,InvoiceDate,Year,Month,MonthName,DayOfWeek,Hour
0,2010-12-01 08:26:00,2010,12,December,Wednesday,8
1,2010-12-01 08:26:00,2010,12,December,Wednesday,8
2,2010-12-01 08:26:00,2010,12,December,Wednesday,8
3,2010-12-01 08:26:00,2010,12,December,Wednesday,8
4,2010-12-01 08:26:00,2010,12,December,Wednesday,8


In [5]:
print("=== BEFORE CLEANING ===")
print(f"Total rows: {len(df):,}")
print(f"Missing CustomerIDs: {df['CustomerID'].isnull().sum():,}")
print(f"Missing Descriptions: {df['Description'].isnull().sum():,}")

# Save a copy with ALL data including missing CustomerIDs
df_all = df.copy()

# Clean version — only rows with CustomerIDs
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print("\n=== AFTER CLEANING ===")
print(f"Rows with CustomerID: {len(df_clean):,}")
print(f"Rows dropped: {len(df) - len(df_clean):,}")

=== BEFORE CLEANING ===
Total rows: 541,909
Missing CustomerIDs: 135,080
Missing Descriptions: 1,454

=== AFTER CLEANING ===
Rows with CustomerID: 406,829
Rows dropped: 135,080


In [6]:
# Returns have invoice numbers starting with 'C'
# and negative quantities
df_clean['IsReturn'] = df_clean['InvoiceNo'].astype(str).str.startswith('C')

returns = df_clean[df_clean['IsReturn'] == True]
purchases = df_clean[df_clean['IsReturn'] == False]

print(f"Total transactions: {len(df_clean):,}")
print(f"Returns: {len(returns):,} ({len(returns)/len(df_clean)*100:.1f}%)")
print(f"Purchases: {len(purchases):,} ({len(purchases)/len(df_clean)*100:.1f}%)")

Total transactions: 406,829
Returns: 8,905 (2.2%)
Purchases: 397,924 (97.8%)


In [7]:
# Remove rows with bad data
purchases = purchases[purchases['UnitPrice'] > 0]
purchases = purchases[purchases['Quantity'] > 0]

# Create revenue column
purchases['Revenue'] = purchases['Quantity'] * purchases['UnitPrice']

print(f"Total Revenue: £{purchases['Revenue'].sum():,.2f}")
print(f"Average order value: £{purchases['Revenue'].mean():.2f}")

Total Revenue: £8,911,407.90
Average order value: £22.40


In [8]:
purchases.to_csv('../data/clean_data.csv', index=False)

print("Clean data saved successfully")
print(f"Final dataset rows: {len(purchases):,}")
print(f"Final dataset columns: {purchases.shape[1]}")

Clean data saved successfully
Final dataset rows: 397,884
Final dataset columns: 15


In [9]:
print("=" * 50)
print("UK RETAIL DATA CLEANING SUMMARY")
print("=" * 50)

print(f"Original rows: {len(df):,}")
print(f"Rows after customer cleaning: {len(df_clean):,}")
print(f"Returns removed: {len(returns):,}")
print(f"Final clean purchase rows: {len(purchases):,}")

print("\nKEY BUSINESS METRICS")
print(f"Total Revenue: £{purchases['Revenue'].sum():,.2f}")
print(f"Average Order Item Value: £{purchases['Revenue'].mean():.2f}")
print(f"Unique Customers: {purchases['CustomerID'].nunique():,}")
print(f"Countries: {purchases['Country'].nunique()}")

UK RETAIL DATA CLEANING SUMMARY
Original rows: 541,909
Rows after customer cleaning: 406,829
Returns removed: 8,905
Final clean purchase rows: 397,884

KEY BUSINESS METRICS
Total Revenue: £8,911,407.90
Average Order Item Value: £22.40
Unique Customers: 4,338
Countries: 37
